# 15 · Does the citation actually support the claim?

## The problem

LLM answers (and plenty of humans) attach citations that **mention the topic** but don't establish the fact, or even
say the opposite. A footnote looks like evidence, so readers trust it. Typical failures:
- the source is about the right thing but never says the claimed fact (*"bike-share reduced car traffic by 10%"*),
- the source says the **opposite** (*"the program became profitable"* when it says it runs at a loss),
- the "quote" isn't in the source at all: the model **invented** it.

## Why Jev

Checking *does this passage support this claim?* is a judgment call about meaning, and you want it on **every**
citation in every answer, so it must be fast and cheap. Jev is a **System One** model: a typed `Choice` with
**calibrated probabilities**, in about 100 ms, for a fraction of a cent.

But not every part of this needs a model. *Is the quoted text literally in the source?* is a string comparison, and
code does that **exactly**, for free. The pattern: **split the judgment (Jev) from the verification you can do exactly
(code).**

## What we'll build

```
(claim, cited passage, quote) ──┬─► code:  is the quote really in the passage?  ── no ──► FABRICATED QUOTE
                                │
                                └─► Jev:   Choice  supports / contradicts / not_addressed
                                                      │
                                            plain-Python policy ──► OK · CONTRADICTED · UNSUPPORTED
```

## What you'll learn
- A three-way `Choice` with an explicit **"neither"** option (`not_addressed`), and how to read `.choice`,
  `.confidence` and `.probabilities`.
- Combining an exact, deterministic check with a model judgment, and which one wins when they disagree.
- How precise wording in a claim (*"legally required"*) changes the verdict.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #8.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the Jev client, `jev`, that every later cell uses,
- imports the Jev question types (here we only need `Choice`; `re` is used for the exact quote check),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so you can *see* the numbers.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The question

One `Choice` with three labels. Each label has a one-line definition, and those definitions are the model's whole
world, so they are worded carefully:

| label | meaning | what it catches |
|---|---|---|
| `supports` | the passage **establishes** the claim is true | a good citation |
| `contradicts` | the passage says the claim is **false** | a citation that argues against you |
| `not_addressed` | the passage is on the topic but neither establishes nor refutes the claim | the most common failure: topic match, no evidence |

**How a `Choice` works:** `.choice` is the winning label, `.confidence` is how sure Jev is, and `.probabilities` is a
dict with one probability per label.

The `state` will be a dict `{"claim": ..., "passage": ...}`; the backticks in the instruction refer to those keys.
This cell only **defines** the question.

In [2]:
CITATION_QUESTION = {"verdict": Choice(
    instructions="What is the relationship between `passage` and `claim`?",
    criteria={
        "supports": "The passage establishes that the claim is true",
        "contradicts": "The passage says the claim is false",
        "not_addressed": "The passage mentions the topic but does not establish or refute the claim",
    },
)}

SOURCE_TEXT = ("In 2023 the city's bike-share fleet grew to 4,000 bicycles. Ridership rose 18% year over year, "
               "although the program still operates at a loss. Helmet use is encouraged but not required.")
print("question defined; source passage has", len(SOURCE_TEXT.split()), "words")

question defined; source passage has 31 words


## Step 2 · Ask once and read the raw answer

One claim that sounds plausible but is not in the source: the passage talks about bike-share, but says nothing about
car traffic.

In [3]:
v = jev.system_one(state={"claim": "Bike-share reduced car traffic downtown by 10%.", "passage": SOURCE_TEXT},
                   questions=CITATION_QUESTION).choices["verdict"]

print(f"choice: {v.choice}   (confidence {v.confidence:.2f})\n")
for label, p in v.probabilities.items():
    print(f"  {label:<14} {bar(p)} {p:.2f}")

choice: not_addressed   (confidence 1.00)

  supports       ···················· 0.00
  contradicts    ···················· 0.00
  not_addressed  ████████████████████ 1.00


**What to notice**
- The verdict is `not_addressed`, with high confidence. The passage is **about** bike-share, but it never says anything
  about car traffic, so the citation is decoration, not evidence.
- That's the failure a human skimming the footnote misses: topic overlap looks like support.
- You get the full distribution, so you could also flag "supports, but only at 0.6" for review.

## Step 3 · The exact check, no model needed

When an answer quotes its source, code can verify the quote **exactly**. `norm()` lower-cases text, collapses repeated
whitespace and trims quote marks and full stops, so harmless formatting differences don't count as fabrication. Then
`quote_in_source()` is a plain substring test. No quote at all counts as `True` (nothing to verify).

The two test lines show it tolerates a double space, but not a changed number.

In [4]:
def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip(" .\"'")


def quote_in_source(quote, source):
    return not quote or norm(quote) in norm(source)


print(quote_in_source("rose 4.1% in 2023", "Revenue rose 4.1%  in 2023, driven by services."))   # True: just a double space
print(quote_in_source("rose 9% in 2023",   "Revenue rose 4.1% in 2023, driven by services."))    # False: the number changed

True
False


## Step 4 · Combine both into one checker

`check_citation()` runs both checks and turns them into one status. The rules, checked in order:

| order | condition | status |
|---|---|---|
| 1 | the quote is **not** in the source | **FABRICATED QUOTE**, whatever Jev says. The exact check wins. |
| 2 | Jev says `supports` with confidence ≥ 0.7 | **OK** |
| 3 | Jev says `contradicts` | **CONTRADICTED** |
| 4 | anything else (`not_addressed`, or a weak `supports`) | **UNSUPPORTED** |

The exact check goes first because it is **certain**: a made-up quote is a hard failure no matter how plausible the
claim is. The guard at the top gives a clear message if the kernel restarted and forgot `CITATION_QUESTION`.
This cell only defines the function.

In [5]:
SUPPORT_MIN = 0.7   # a 'supports' verdict below this confidence is treated as unsupported


def check_citation(claim, passage, quote, client):
    if "CITATION_QUESTION" not in globals():
        raise RuntimeError("CITATION_QUESTION is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    quote_ok = quote_in_source(quote, passage)
    c = client.system_one(state={"claim": claim, "passage": passage}, questions=CITATION_QUESTION).choices["verdict"]
    if not quote_ok:
        status = "FABRICATED QUOTE"
    elif c.choice == "supports" and c.confidence >= SUPPORT_MIN:
        status = "OK"
    elif c.choice == "contradicts":
        status = "CONTRADICTED"
    else:
        status = "UNSUPPORTED"
    return {"status": status, "verdict": c.choice, "conf": c.confidence, "probs": c.probabilities, "quote_ok": quote_ok}


print("check_citation() defined")

check_citation() defined


## Step 5 · Run it on five citations

Five `(claim, quote)` pairs, all citing the same bike-share passage, designed to hit every status:

In [6]:
CITES = [
    ("The bike-share program expanded to 4,000 bikes in 2023.", "fleet grew to 4,000 bicycles"),
    ("The bike-share program became profitable in 2023.", "operates at a loss"),
    ("Bike-share riders are legally required to wear helmets.", "Helmet use is encouraged"),
    ("Bike-share reduced car traffic downtown by 10%.", "Ridership rose 18%"),
    ("Ridership doubled in 2023.", "Ridership doubled year over year"),
]
for claim, quote in CITES:
    d = check_citation(claim, SOURCE_TEXT, quote, jev)
    print(f"[{d['status']:^16}] {claim}")
    print(f"      jev: {d['verdict']} ({d['conf']:.2f})   quote found in source: {d['quote_ok']}")

[       OK       ] The bike-share program expanded to 4,000 bikes in 2023.
      jev: supports (1.00)   quote found in source: True


[  CONTRADICTED  ] The bike-share program became profitable in 2023.
      jev: contradicts (1.00)   quote found in source: True


[  CONTRADICTED  ] Bike-share riders are legally required to wear helmets.
      jev: contradicts (1.00)   quote found in source: True


[  UNSUPPORTED   ] Bike-share reduced car traffic downtown by 10%.
      jev: not_addressed (1.00)   quote found in source: True


[FABRICATED QUOTE] Ridership doubled in 2023.
      jev: contradicts (1.00)   quote found in source: False


**What to notice**
- **4,000 bikes → OK.** The passage states it directly, and the quote is real.
- **Profitable → CONTRADICTED.** The quote is real (*"operates at a loss"*), and it argues *against* the claim. A real
  quote is not the same as a supporting quote.
- **Helmets legally required → CONTRADICTED.** This one is worth pausing on. You might expect `not_addressed`, since
  the passage never mentions *law*. But it says helmet use is *"encouraged but **not required**"*, which directly
  denies that riders are required to wear one. So `contradicts` is the defensible reading. The quote the claim chose
  (*"Helmet use is encouraged"*) is real but cherry-picked: it stops just before the words that refute the claim.
- **Car traffic → UNSUPPORTED.** The quote is real but about ridership; the passage never addresses traffic.
- **Ridership doubled → FABRICATED QUOTE.** *"Ridership doubled year over year"* isn't in the source, so code flags it
  before Jev's verdict matters. (Jev independently says `contradicts`, since 18% is not doubling. Two checks, same
  conclusion, but the exact one is the one you can prove.)

## Step 6 · Give it a face

Everything from here on is **presentation only**. This cell is a small style kit (colours, `box`, `badge`, `meter`) and
`form_app()`, which wires text boxes and a button to any function that returns HTML. `citation_html()` shows one citation as a card: status, the quote check, and Jev's three probabilities.
**The decision logic above is untouched**: the UI only calls the same functions you've already seen run.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

The next cell defines `citation_html()` and renders all five citations from Step 5 as cards. Same function, same
results, just easier to read in a room.

In [8]:
CIT_COLOR = {"OK": SAGE, "UNSUPPORTED": AMBER, "CONTRADICTED": RED, "FABRICATED QUOTE": RED}


def citation_html(claim, passage, quote):
    d = check_citation(claim, passage, quote, jev)
    probs = "".join(meter(k, v) for k, v in d["probs"].items())
    q = (f'<div style="font-size:12px;color:{SAGE if d["quote_ok"] else RED};">quote "{quote}" '
         f'{"found" if d["quote_ok"] else "NOT found"} in source</div>') if quote else ""
    return box(f'<div style="{CARD}border-left:5px solid {CIT_COLOR[d["status"]]};">'
               f'{badge(d["status"], CIT_COLOR[d["status"]])} <b>{claim}</b>{q}'
               f'<div style="margin-top:8px;">{probs}</div></div>')


display(HTML(box("".join(citation_html(c, SOURCE_TEXT, q) for c, q in CITES))))

## Step 7 · Try it yourself

Edit the claim, the cited passage and (optionally) a quoted snippet, then press **Check** (needs a live kernel). Ideas,
all against the default passage:
- `The program operates at a profit.` (no quote) → **CONTRADICTED**.
- `Helmets are optional for bike-share riders.` → **OK**: the mirror image of the "legally required" claim.
- `Ridership rose by about a fifth in 2023.` → Jev accepts 18% as "about a fifth": **OK**. Is that rounding acceptable
  in *your* domain? If not, tighten the `supports` definition.
- `Ridership rose 18% in 2023.` with quote `Ridership rose 20%` → **FABRICATED QUOTE**: the claim is true, but the quote
  was altered.

In [9]:
form_app([("claim", "Claim", "The program operates at a profit.", 1),
          ("passage", "Cited passage", SOURCE_TEXT, 4),
          ("quote", "Quoted text (optional)", "", 1)], citation_html, "Check")

Output()

## Recap and exercises

**Key ideas**
1. **Topic overlap is not evidence.** The `not_addressed` label is what catches citations that are merely on-topic.
2. **Split judgment from verification.** Jev judges meaning; code verifies quotes exactly. When the exact check fails,
   it wins.
3. **Precise words matter.** *"Legally required"* vs *"encouraged but not required"* is a contradiction, not a gap. Read
   the claim as literally as the model does.
4. **The model measures, your code decides.** `SUPPORT_MIN` and the order of the rules are ordinary Python.

**Exercises**
- Raise `SUPPORT_MIN` to 0.95 and try *"Ridership rose by about a fifth"*. Does it still pass?
- Add a fourth label, `partially_supports` (*"supports part of the claim but not all of it"*), and try
  *"The fleet grew to 4,000 bikes and the program became profitable."*
- Make `quote_in_source()` tolerant of curly quotes and dashes (`“ ” – —`) and test it.
- Split a long LLM answer into sentences and run `check_citation()` on each one against its cited passage.